# Phase 2.3–2.4 — Fine-tune all 5 models x 3 seeds

Runs `{mbert, muril, indic-bert, hing-bert, hing-roberta}` x seeds `{42, 43, 44}` = **15 runs**,
each producing the load-bearing output: one per-sentence CSV in `preds/`
(`sentence_id, model, seed, true_label, pred_label, correct`), all on the **same fixed test split**
from the master so Phase 4 can compare models sentence-by-sentence.

**Built for Colab reality:**
- **Resumable at the run level** — each run skips if its `preds/{model}_seed{seed}.csv` already exists.
  If Colab disconnects, just re-run the loop cell; finished runs are skipped, only the interrupted/pending
  ones re-execute. This replaces per-epoch Drive checkpointing (15 x 438 MB would bloat Drive and slow I/O).
- Checkpoints go to **local disk** (`/content/ckpt_tmp`) and are deleted after each run's CSV is safely on Drive
  (flip `KEEP_CHECKPOINTS` to keep them). The tiny CSVs on Drive are the real safety net.
- **Per-run try/except** — IndicBERT gating or one bad run won't kill the other 14.
- No `tokenizer=` arg to `Trainer` (removed in newer transformers); dynamic padding via `DataCollatorWithPadding`.

**Prereq:** `data/sa_master.parquet` + `data/label_mapping.json` from Phase 2.1.

⏱️ Full 15-run sweep is roughly a few hours on a T4. **Smoke-test tip:** set `SEEDS = [42]` first to
sweep all 5 models quickly (~1 hr), confirm the CSVs and summary look right, then set it back to
`[42, 43, 44]` and re-run — the seed-42 runs are skipped automatically.


In [ ]:
!pip -q install "transformers>=4.41" datasets accelerate scikit-learn pandas pyarrow sentencepiece

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# IndicBERT (ai4bharat/indic-bert) is gated: needs an accepted licence + token.
# Store your token as a Colab secret named HF_TOKEN (key icon in the left sidebar), then run this.
try:
    from google.colab import userdata
    from huggingface_hub import login
    login(userdata.get("HF_TOKEN"))
    print("HF login OK")
except Exception as e:
    print("HF login skipped/failed:", e)
    print("-> IndicBERT will error until HF_TOKEN is set + terms accepted on its model page.")
    print("-> The other 4 models still run fine; the loop catches IndicBERT's failure and continues.")

In [ ]:
import os, gc, json, shutil, glob, numpy as np, pandas as pd, torch
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding, set_seed)
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score

PROJECT    = "/content/drive/MyDrive/codemix_project"
DATA       = f"{PROJECT}/data"
PREDS      = f"{PROJECT}/preds"
RESULTS    = f"{PROJECT}/results"
LOCAL_CKPT = "/content/ckpt_tmp"          # local disk; wiped on disconnect (that's fine)
for d in (PREDS, RESULTS, LOCAL_CKPT):
    os.makedirs(d, exist_ok=True)

MODELS = {
    "mbert":        "bert-base-multilingual-cased",
    "muril":        "google/muril-base-cased",
    "indic-bert":   "ai4bharat/indic-bert",     # gated + expected to score lower (by design)
    "hing-bert":    "l3cube-pune/hing-bert",
    "hing-roberta": "l3cube-pune/hing-roberta",
}
SEEDS   = [42, 43, 44]        # smoke test: set to [42] first, then expand
MAX_LEN = 128
KEEP_CHECKPOINTS = False      # delete each run's weights after its CSV is written (saves disk)

df = pd.read_parquet(f"{DATA}/sa_master.parquet")
with open(f"{DATA}/label_mapping.json") as f:
    label2id = json.load(f)
id2label = {int(v): k for k, v in label2id.items()}

assert {"sentence_id", "text", "label_id", "split"}.issubset(df.columns), "master schema drift"
assert df["sentence_id"].is_unique, "sentence_id must be unique"

train_df = df[df.split == "train"].reset_index(drop=True)
val_df   = df[df.split == "val"].reset_index(drop=True)
test_df  = df[df.split == "test"].reset_index(drop=True)
print(f"train={len(train_df)}  val={len(val_df)}  test={len(test_df)}")
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu")

In [ ]:
def tokenize_all(tok):
    def to_ds(frame):
        ds = Dataset.from_pandas(frame[["sentence_id", "text", "label_id"]], preserve_index=False)
        ds = ds.map(lambda b: tok(b["text"], truncation=True, max_length=MAX_LEN), batched=True)
        return ds.rename_column("label_id", "labels")
    return to_ds(train_df), to_ds(val_df), to_ds(test_df)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"accuracy": accuracy_score(labels, preds),
            "f1_macro": f1_score(labels, preds, average="macro")}

def run_one(model_name, model_id, seed, train_ds, val_ds, test_ds, collator):
    csv_path = f"{PREDS}/{model_name}_seed{seed}.csv"
    if os.path.exists(csv_path):
        print(f"  skip (exists): {model_name} seed {seed}")
        return
    set_seed(seed)
    model = AutoModelForSequenceClassification.from_pretrained(
        model_id, num_labels=3, id2label=id2label, label2id=label2id)
    args = TrainingArguments(
        output_dir=f"{LOCAL_CKPT}/{model_name}_seed{seed}",
        num_train_epochs=3,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        learning_rate=2e-5,
        weight_decay=0.01,
        warmup_steps=100,
        eval_strategy="epoch",
        save_strategy="epoch",
        load_best_model_at_end=True,
        metric_for_best_model="f1_macro",
        greater_is_better=True,
        save_total_limit=1,
        logging_steps=100,
        fp16=torch.cuda.is_available(),
        seed=seed,
        report_to="none",
    )
    trainer = Trainer(                       # no tokenizer= arg -> padding via data_collator
        model=model, args=args,
        train_dataset=train_ds, eval_dataset=val_ds,
        data_collator=collator, compute_metrics=compute_metrics)
    trainer.train()

    preds = np.argmax(trainer.predict(test_ds).predictions, axis=-1)
    out = pd.DataFrame({
        "sentence_id": test_df["sentence_id"].values,   # unchanged from master
        "model": model_name, "seed": seed,
        "true_label": test_df["label_id"].values,       # int ids; decode via label_mapping.json in Phase 4
        "pred_label": preds,
    })
    out["correct"] = (out.true_label == out.pred_label).astype(int)
    assert len(out) == len(test_df) and out["sentence_id"].is_unique, "key discipline violated"
    out.to_csv(csv_path, index=False)
    f1 = f1_score(test_df["label_id"], preds, average="macro")
    print(f"  {model_name} seed {seed}: test macro-F1={f1:.4f}  ->  {csv_path}")

    del model, trainer
    gc.collect(); torch.cuda.empty_cache()
    if not KEEP_CHECKPOINTS:
        shutil.rmtree(args.output_dir, ignore_errors=True)

In [ ]:
# --- main sweep: resumable, fault-tolerant ------------------------------------
for model_name, model_id in MODELS.items():
    pending = [s for s in SEEDS if not os.path.exists(f"{PREDS}/{model_name}_seed{s}.csv")]
    if not pending:
        print(f"{model_name}: all seeds done, skipping")
        continue
    print(f"\n=== {model_name}  ({model_id})  | pending seeds {pending} ===")
    try:
        tok = AutoTokenizer.from_pretrained(model_id)
        train_ds, val_ds, test_ds = tokenize_all(tok)   # tokenized once, reused across this model's seeds
        collator = DataCollatorWithPadding(tok)
    except Exception as e:
        print(f"  !! could not load/tokenize {model_name}: {e}")
        continue
    for seed in SEEDS:
        try:
            run_one(model_name, model_id, seed, train_ds, val_ds, test_ds, collator)
        except Exception as e:
            print(f"  !! {model_name} seed {seed} FAILED: {e}")
    del tok, train_ds, val_ds, test_ds, collator
    gc.collect(); torch.cuda.empty_cache()

print("\nloop complete (re-run this cell after any disconnect to finish pending runs)")

In [ ]:
# --- summary: rebuilt from the CSVs on disk, so it works even across sessions --
rows = []
for path in sorted(glob.glob(f"{PREDS}/*_seed*.csv")):
    p = pd.read_csv(path)
    rows.append({
        "model": p["model"].iloc[0],
        "seed": int(p["seed"].iloc[0]),
        "accuracy": p["correct"].mean(),
        "f1_macro": f1_score(p["true_label"], p["pred_label"], average="macro"),
        "n": len(p),
    })

runs = pd.DataFrame(rows).sort_values(["model", "seed"])
print("per-run:")
print(runs.to_string(index=False))

summary = (runs.groupby("model")
                .agg(f1_mean=("f1_macro", "mean"), f1_std=("f1_macro", "std"),
                     acc_mean=("accuracy", "mean"), acc_std=("accuracy", "std"),
                     seeds=("seed", "count"))
                .reset_index()
                .sort_values("f1_mean", ascending=False)
                .round(4))
summary.to_csv(f"{RESULTS}/sa_finetune_summary.csv", index=False)
print("\n=== macro-F1 mean +/- std over seeds ===")
print(summary.to_string(index=False))
print(f"\nsaved -> {RESULTS}/sa_finetune_summary.csv")

## What to check

- **15 CSVs** in `preds/` (or however many `len(MODELS) * len(SEEDS)` is), each with the same row count as the test split.
- Hinglish models (`hing-bert`, `hing-roberta`) at the top of the summary; **IndicBERT lower is expected**,
  only a ~33 F1 collapse signals a real bug.
- `results/sa_finetune_summary.csv` written with mean ± std per model.

## Next — Phase 3 (feature extraction)

Build `features/sentence_features.csv` keyed by `sentence_id`: CMI (already in the master),
switch_density (from `lang_tags`), length, hi_ratio, oov_rate, caps_ratio. Join onto each prediction
CSV on `sentence_id` and assert row counts — that join is what Phase 4's correlation and
robustness-gap analysis run on.
